# yoctogpt
> a GPT, from scratch, in pure Python: no PyTorch, no NumPy, nothing to install

Run the cells top to bottom. By the end, ~4,000 random numbers learn to speak. The compressed 100-line version lives in [`v1.py`](https://github.com/Infatoshi/yoctogpt/blob/master/v1.py): read it at the end, as a victory lap.

![Full GPT Model](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/06-full-gpt.png)

## 1 · Data & Tokenization

In [ ]:
import math
import random

random.seed(67)

txt = "Peter Piper picked a peck of pickled peppers.\nA peck of pickled peppers Peter Piper picked.\nIf Peter Piper picked a peck of pickled peppers,\nWhere's the peck of pickled peppers Peter Piper picked?\n"
print(len(txt))

In [ ]:
characters = sorted(set(txt))   # every unique character, in a fixed order
vocab_size = len(characters)

print(f"vocab size: {vocab_size}")
print(characters)

In [ ]:
token_ids = []
for char in txt:
    token_ids.append(characters.index(char))   # each character becomes a number

print(f"first 20 tokens: {token_ids[:20]}")

In [ ]:
split_point = int(len(token_ids) * 0.9)
train_data = token_ids[:split_point]   # the model learns from this 90%
val_data = token_ids[split_point:]     # and is tested on this unseen 10%

print(f"train: {len(train_data)} tokens, val: {len(val_data)} tokens")

In [ ]:
# TRY IT: encode your own text, decode it back
# my_ids = []
# for c in "Pepper":
#     my_ids.append(characters.index(c))
# print(my_ids)
# decoded = ""
# for i in my_ids:
#     decoded = decoded + characters[i]
# print(decoded)

## Settings (hyperparameters)

In [ ]:
context_length = 8     # the model sees at most the last 8 characters
embed_dim = 16         # each token becomes 16 numbers
learning_rate = 0.01   # how big a nudge each update takes
init_std = 0.08        # size of the starting random weights (we test this number in section 9)

## 2 · Embeddings: a list of numbers for every character

A token ID like `8` is just a name tag. The model needs something it can do math with, so every character gets its own row of
16 numbers. They start random; training is what will make them mean something.

![Token Embedding](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/03-token-embedding.png)

In [ ]:
#@title Value: a number that keeps a receipt (run this; we open it up in section 7)
# For now, read Value(0.12) as "the number 0.12". .data gets the plain number back.
# Every time two Values are added or multiplied, the result also writes down how it was made.
# The model ignores that receipt until it has to learn, in section 7.

tape = []   # every Value signs in here at birth, in creation order

class Value:
    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # the actual number
        self.grad = 0.0                 # its share of the blame, filled in by backward()
        self.children = children        # the Values this one was made from
        self.local_grads = local_grads  # how much this moves per nudge of each child
        tape.append(self)

    def __add__(self, other):           # a + b    (both must be Values)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):           # a * b    (both must be Values)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, n):               # a ** n   (n is a plain number)   used by normalize and softmax
        return Value(self.data ** n, (self,), (n * self.data ** (n - 1),))

    def exp(self):                      # e ** a   used by softmax
        result = math.exp(self.data)
        return Value(result, (self,), (result,))

    def log(self):                      # ln(a)    used by the loss
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def relu(self):                     # max(0, a)   used by the feed-forward
        if self.data > 0:
            slope = 1.0
        else:
            slope = 0.0
        return Value(max(0.0, self.data), (self,), (slope,))

    def backward(self):
        self.grad = 1.0                  # the loss blames itself 100%
        for node in reversed(tape):      # newest first: a result is always younger than its ingredients
            for i in range(len(node.children)):
                node.children[i].grad += node.local_grads[i] * node.grad   # chain rule

In [ ]:
# one row of 16 random numbers for each of the 24 characters
token_embed = []
for row_number in range(vocab_size):
    row = []
    for column in range(embed_dim):
        row.append(Value(random.gauss(0, init_std)))   # small and random
    token_embed.append(row)

p_row = token_embed[characters.index("P")]   # 'P' is token 8
print(f"token_embed is {len(token_embed)} rows x {len(token_embed[0])} numbers")
print("the row for 'P':")
for v in p_row:
    print(round(v.data, 3), end="  ")

Attention (section 4) looks at every earlier character at once, so it has no idea which came first. We fix that the same way:
a second table, one row per **slot** in the 8-character window.

In [ ]:
# one row of 16 random numbers for each of the 8 slots
position_embed = []
for slot in range(context_length):
    row = []
    for column in range(embed_dim):
        row.append(Value(random.gauss(0, init_std)))
    position_embed.append(row)

print(f"position_embed is {len(position_embed)} rows x {len(position_embed[0])} numbers")
print("the row for slot 0:")
for v in position_embed[0]:
    print(round(v.data, 3), end="  ")

What goes into the model is the two rows added together, number by number: **who am I + where am I**.

![Token plus position](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/12-embed-add.png)

In [ ]:
# 'P' sitting in slot 0
tok_row = token_embed[characters.index("P")]
pos_row = position_embed[0]

combined = []
for j in range(embed_dim):
    combined.append(tok_row[j] + pos_row[j])

for j in range(4):
    print(f"{tok_row[j].data:+.3f}  +  {pos_row[j].data:+.3f}  =  {combined[j].data:+.3f}")
print("... (16 numbers in all)")

In [ ]:
#@title ▶ Playground: pick a character and a slot
# (the sliders come from ipywidgets, which Colab already has; the math below is plain Python)
def show_embedding(character="P", slot=0):
    tok_row = token_embed[characters.index(character)]
    pos_row = position_embed[slot]
    print(f"{'':>4}{'token':>8}   {'position':>8}   {'combined':>8}")
    for j in range(embed_dim):
        t, p = tok_row[j].data, pos_row[j].data
        bar = ("█" * round(abs(t + p) * 60)) if t + p > 0 else ("░" * round(abs(t + p) * 60))
        print(f"{j:>4}{t:+8.3f} + {p:+8.3f} = {t + p:+8.3f}  {bar}")
    print("\n█ positive, ░ negative")

try:
    import ipywidgets
    ipywidgets.interact(show_embedding, character=[(repr(c), c) for c in characters], slot=(0, context_length - 1))
except ImportError:
    show_embedding()

We just wrote the same double loop twice. Give it a name, because the rest of the model is more tables like these:

In [ ]:
def make_matrix(rows, cols):
    matrix = []
    for r in range(rows):
        row = []
        for c in range(cols):
            row.append(Value(random.gauss(0, init_std)))   # small random start
        matrix.append(row)
    return matrix

## 3 · Three Tools: linear, normalize, softmax

![Matrix times Vector](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/08-matrix-vector.png)

In [ ]:
def linear(input_vector, weight_matrix):
    output = []
    for row in weight_matrix:          # one output number per row
        dot_product = Value(0)
        for j in range(len(input_vector)):
            dot_product = dot_product + row[j] * input_vector[j]
        output.append(dot_product)
    return output

In [ ]:
def normalize(vector):
    sum_sq = Value(0)
    for x in vector:
        sum_sq = sum_sq + x * x
    mean_squared = sum_sq * Value(1 / len(vector))
    scale = (mean_squared + Value(1e-5)) ** -0.5    # 1 / sqrt(mean of squares)
    result = []
    for x in vector:
        result.append(x * scale)             # same direction, typical size ~1
    return result

In [ ]:
def softmax(scores):
    max_val = scores[0].data                 # find the biggest score...
    for v in scores:
        if v.data > max_val:
            max_val = v.data                 # ...so we can subtract it: exp() can't overflow
    exp_scores = []
    for x in scores:
        exp_scores.append((x + Value(-max_val)).exp())
    total = Value(0)
    for x in exp_scores:
        total = total + x
    probs = []
    for x in exp_scores:
        probs.append(x * (total ** -1))      # divide by the sum: now they sum to 1
    return probs

In [ ]:
test = softmax([Value(1), Value(2), Value(3)])
total = 0.0
for p in test:
    print(round(p.data, 4))
    total = total + p.data
print("sum:", total)

In [ ]:
#@title ▶ Playground: drag three scores, watch them become probabilities
def show_softmax(score_a=1.0, score_b=2.0, score_c=3.0):
    scores = [score_a, score_b, score_c]
    biggest = max(scores)
    exps = []
    for s in scores:
        exps.append(math.exp(s - biggest))    # subtract the biggest first, like softmax() does
    total = sum(exps)
    for name, s, e in zip(["a", "b", "c"], scores, exps):
        p = e / total
        print(f"{name}: score {s:+5.1f}   exp {e:6.3f}   / {total:6.3f}  =  {p:5.3f}  {'█' * round(p * 40)}")
    print(f"sum of probabilities: {sum(e / total for e in exps):.3f}")

try:
    import ipywidgets
    ipywidgets.interact(show_softmax, score_a=(-5.0, 5.0, 0.1), score_b=(-5.0, 5.0, 0.1), score_c=(-5.0, 5.0, 0.1))
except ImportError:
    show_softmax()

In [ ]:
# PROOF: two linear layers with no relu collapse into one linear layer
W1 = [[0.5, -1.0], [2.0, 1.0]]
W2 = [[1.0, 1.0], [-1.0, 0.5]]

def matvec(W, v):
    out = []
    for row in W:
        total = 0.0
        for j in range(len(v)):
            total = total + row[j] * v[j]
        out.append(total)
    return out

v = [3.0, -2.0]
two_layers = matvec(W2, matvec(W1, v))

W_combined = [[0.0, 0.0], [0.0, 0.0]]
for i in range(2):
    for j in range(2):
        for k in range(2):
            W_combined[i][j] = W_combined[i][j] + W2[i][k] * W1[k][j]
one_layer = matvec(W_combined, v)

print(f"two linear layers: {two_layers}")
print(f"one combined layer: {one_layer}")
# identical, for EVERY input. depth without relu's bend is fake depth.

## 4 · The Forward Pass

tokens in → a score for every possible next character out

![Attention Mechanism](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/04-attention.png)

![Attention one token](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/11-attention-one-token.png)

In [ ]:
# the rest of the brain: more tables of small random numbers
w_query    = make_matrix(embed_dim, embed_dim)            # attention: what am I looking for?
w_key      = make_matrix(embed_dim, embed_dim)            # attention: what do I contain?
w_value    = make_matrix(embed_dim, embed_dim)            # attention: what do I pass on?
w_attn_out = make_matrix(embed_dim, embed_dim)

ff_up   = make_matrix(embed_dim * 4, embed_dim)           # 64 x 16: room to think
ff_down = make_matrix(embed_dim, embed_dim * 4)           # 16 x 64: back down

output_proj = make_matrix(vocab_size, embed_dim)          # 24 x 16: back to character scores

In [ ]:
def gpt_forward(token_ids, return_attention=False):
    keys = []     # the cache: one key per past token
    values = []   # one value per past token
    attention_maps = []
    all_scores = []

    for pos in range(len(token_ids)):

        # who am I + where am I
        tok_emb = token_embed[token_ids[pos]]
        pos_emb = position_embed[pos]
        combined = []
        for j in range(embed_dim):
            combined.append(tok_emb[j] + pos_emb[j])
        x = normalize(combined)

        # ---- attention ----
        residual = x
        x = normalize(x)
        query = linear(x, w_query)
        key = linear(x, w_key)
        value = linear(x, w_value)
        keys.append(key)
        values.append(value)

        # score every past token: one number per token t
        scores = []
        for t in range(pos + 1):
            dot = Value(0)
            for j in range(embed_dim):
                dot = dot + query[j] * keys[t][j]
            scores.append(dot * Value(embed_dim ** -0.5))   # divide by sqrt(16) = 4 for stability
        attn_weights = softmax(scores)                 # blend weights, sum to 1
        if return_attention:
            weights_now = []
            for w in attn_weights:
                weights_now.append(w.data)
            attention_maps.append(weights_now)

        # blend the values: one number per dimension j
        attended = []
        for j in range(embed_dim):
            blended = Value(0)
            for t in range(pos + 1):
                blended = blended + attn_weights[t] * values[t][j]
            attended.append(blended)

        attn_out = linear(attended, w_attn_out)
        new_x = []
        for j in range(embed_dim):
            new_x.append(attn_out[j] + residual[j])    # skip connection
        x = new_x

        # ---- feed-forward ----
        ff_in = normalize(x)
        ff_hidden = []
        for h in linear(ff_in, ff_up):
            ff_hidden.append(h.relu())                 # the bend
        ff_out = linear(ff_hidden, ff_down)
        new_x = []
        for j in range(embed_dim):
            new_x.append(ff_out[j] + x[j])             # skip connection again
        x = new_x

        # 24 raw scores, one per character ("logits")
        all_scores.append(linear(x, output_proj))

    if return_attention:
        return all_scores, attention_maps
    return all_scores

### One transformer block. Real models stack 30 to 100 of these.

![GPT Block](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/05-gpt-block.png)

In [ ]:
# every number the model can learn, in one list
all_matrices = [token_embed, position_embed, w_query, w_key, w_value,
                w_attn_out, ff_up, ff_down, output_proj]
all_parameters = []
for matrix in all_matrices:
    for row in matrix:
        for param in row:
            all_parameters.append(param)

print(f"total parameters: {len(all_parameters):,}")

## 5 · Generation

Predict → sample → append → repeat. **This loop is ChatGPT**: when it streams an answer word by word, it is running exactly this, one token at a time.

The weights are still random, so expect gibberish. That is the problem the rest of the notebook solves.

In [ ]:
prompt_ids = []
for c in "Peter Piper ":
    prompt_ids.append(characters.index(c))

In [ ]:
# run this cell again and again: ONE new character per run
logits = gpt_forward(prompt_ids[-context_length:])
probs = softmax(logits[-1])

chances = []
for p in probs:
    chances.append(p.data)
next_token = random.choices(range(vocab_size), weights=chances)[0]
prompt_ids.append(next_token)

text_so_far = ""
for t in prompt_ids:
    text_so_far = text_so_far + characters[t]
print(text_so_far)

In [ ]:
# pressing shift-enter 50 times gets old: wrap it in a loop
def generate_text(prompt="Peter Piper ", num_chars=50, temperature=1.0):
    token_ids = []
    for c in prompt:
        token_ids.append(characters.index(c))
    print(prompt, end="", flush=True)
    for _ in range(num_chars):
        logits = gpt_forward(token_ids[-context_length:])
        last = []
        for score in logits[-1]:
            last.append(score * Value(1 / temperature))   # temperature: <1 safe, >1 wild
        probs = softmax(last)
        chances = []
        for p in probs:
            chances.append(p.data)
        next_token = random.choices(range(vocab_size), weights=chances)[0]
        token_ids.append(next_token)
        print(characters[next_token], end="", flush=True)
    print()

generate_text()   # random weights: expect gibberish

## 6 · Loss: scoring the bet

You'd bet ~90% that `Peter Pi_` continues with `p`. The model makes the same kind of bet for all 24 characters; loss = `-log(probability it gave the right answer)`. Confident and right ≈ 0.1, confident and wrong ≈ 4.6, random guessing = ln(24) ≈ 3.18.

In [ ]:
def compute_loss(token_sequence):
    logits = gpt_forward(token_sequence[:-1])
    total_loss = Value(0)
    for pos in range(len(logits)):
        probs = softmax(logits[pos])
        correct_next = token_sequence[pos + 1]
        total_loss = total_loss + probs[correct_next].log() * Value(-1)
    return total_loss * Value(1 / len(logits))

In [ ]:
print(f"expected for random weights: {math.log(vocab_size):.4f}")
print(f"actual initial loss:        {compute_loss(val_data[:context_length + 1]).data:.4f}")

The loss says *how* wrong the bet was. It doesn't say which of the 3,968 numbers to change, or which way. That's next.

## 7 · Autograd: how a number learns

Nudge one number a tiny bit and watch how much the output moves. That ratio is the number's **slope**: its leverage over the output.

![Computation Graph](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/01-computation-graph.png)

In [ ]:
# nudge by hand:
# If y = x², then dy/dx = 2x. At x=3, that's 6.

x = 3.0
y = x ** 2  # y = 9

# Nudge x by a tiny amount
x_nudged = 3.001
y_nudged = x_nudged ** 2  # 9.006001

# How much did y change per unit change in x?
dy = y_nudged - y       # 0.006001
dx = 0.001
slope = dy / dx          # ≈ 6.0

print(f"Manual gradient: {slope:.4f}")
print(f"Calculus says:   {2 * x:.4f}")  # derivative of x² is 2x

In [ ]:
#@title ▶ Playground: nudge x and measure the slope
def show_nudge(x=3.0, nudge=0.001, power=2):
    y = x ** power
    y_nudged = (x + nudge) ** power
    slope = (y_nudged - y) / nudge
    print(f"x = {x}    y = x**{power} = {y:.4f}")
    print(f"nudge x by {nudge}:  x = {x + nudge:.4f}    y = {y_nudged:.6f}")
    print(f"y moved {y_nudged - y:.6f}, which is {slope:.4f} times the nudge")
    print(f"power rule says {power} * x**{power - 1} = {power * x ** (power - 1):.4f}")

try:
    import ipywidgets
    ipywidgets.interact(show_nudge, x=(-10.0, 10.0, 0.5), nudge=[1.0, 0.1, 0.01, 0.001], power=(1, 4))
except ImportError:
    show_nudge()

Nudging works, but doing it once per parameter would mean 3,968 extra forward passes per step. Instead, every `Value` works its
slope out **once, when it's created**, and writes it on its receipt (`local_grads`). Open the `Value` cell in section 2 (double-click
its title) and read it next to this table:

![Local grads](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/13-local-grads.png)

In [ ]:
#@title ▶ Playground: nudging vs. the receipt
def show_receipt(a=3.0, b=4.0, operation="a * b"):
    if operation == "a + b":
        out = Value(a) + Value(b)
        f = lambda a, b: a + b
    else:
        out = Value(a) * Value(b)
        f = lambda a, b: a * b
    nudge = 0.001
    slope_a = (f(a + nudge, b) - f(a, b)) / nudge
    slope_b = (f(a, b + nudge) - f(a, b)) / nudge
    print(f"{operation} = {out.data}")
    print(f"nudge a: the result moves {slope_a:.3f} times as much")
    print(f"nudge b: the result moves {slope_b:.3f} times as much")
    print(f"the receipt (local_grads) already says: {out.local_grads}")

try:
    import ipywidgets
    ipywidgets.interact(show_receipt, a=(-5.0, 5.0, 0.5), b=(-5.0, 5.0, 0.5), operation=["a + b", "a * b"])
except ImportError:
    show_receipt()

In [ ]:
tape.clear()   # start a fresh tape (the forward passes above left theirs on it)
# quick sanity check: d/dx (x^2) at x=3 should be 6
x = Value(3.0)
y = x ** 2
y.backward()
print(f"x={x.data}, y=x^2={y.data}, dy/dx={x.grad}")

## 8 · Backward: where the blame goes

`x.grad = 6.0`: same answer as the manual nudge, but automatic, for any computation. The whole forward pass is adds, multiplies
and a few `exp`/`log`/`relu`s, and every one of them left a receipt, so `backward()` can walk the tape newest-first and hand each
number its share of the blame.

![One Neuron: where the blame goes](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/10-one-neuron.png)

Run it yourself:

In [ ]:
tape.clear()
# one tiny neuron, backward by hand: y = x * w, and we wanted 10
x = Value(3.0)   # pretend this is the previous layer's output
w = Value(2.0)   # this is ours to train
y = x * w        # y = 6.0

loss = (y + Value(-10)) ** 2   # squared error: (6 - 10)^2 = 16
loss.backward()

print(f"y's blame (dLoss/dy): 2*(6-10) = -8   ('too small: increase me')")
print(f"w.grad = {w.grad}   (= x.data * -8 = 3 * -8)")
print(f"x.grad = {x.grad}   (= w.data * -8 = 2 * -8 -> the message sent leftward)")

# take one training step on w and watch y move toward the target
w.data -= 0.01 * w.grad
print(f"after one nudge: w = {w.data:.2f}, y = x*w = {3.0 * w.data:.2f}  (was 6.0, heading to 10)")

### Chain rule: gradients flow backward through the graph

![Chain Rule](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/02-chain-rule.png)

## 9 · Training

![Training Loop](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/07-training-loop.png)

In [ ]:
# ONE training step. run this cell repeatedly and watch the loss fall.
start = random.randint(0, len(train_data) - context_length - 1)
chunk = train_data[start : start + context_length + 1]

tape.clear()   # fresh sheet of scratch paper: forget last step's graph
loss = compute_loss(chunk)
loss.backward()

for param in all_parameters:
    # RMSprop-style: divide each gradient by its recent typical size
    param.m = 0.99 * getattr(param, "m", 0.0) + 0.01 * (param.grad ** 2)
    param.data = param.data - learning_rate * param.grad / (param.m ** 0.5 + 1e-8)
    param.grad = 0.0

print(f"loss on this chunk: {loss.data:.4f}")

In [ ]:
# same thing, 1000 times
num_steps = 1000
train_losses = []
val_history = []

for step in range(num_steps + 1):
    if step % 100 == 0:
        val_losses = []
        for i in range(0, len(val_data) - context_length, 4):
            val_losses.append(compute_loss(val_data[i : i + context_length + 1]).data)
        val_loss = sum(val_losses) / len(val_losses)
        val_history.append((step, val_loss))
        print(f"\nstep {step}: val loss = {val_loss:.4f}")
        generate_text()

    start = random.randint(0, len(train_data) - context_length - 1)
    chunk = train_data[start : start + context_length + 1]
    tape.clear()   # fresh sheet of scratch paper: forget last step's graph
    loss = compute_loss(chunk)
    train_losses.append(loss.data)
    loss.backward()

    for param in all_parameters:
        param.m = 0.99 * getattr(param, "m", 0.0) + 0.01 * (param.grad ** 2)
        param.data = param.data - learning_rate * param.grad / (param.m ** 0.5 + 1e-8)
        param.grad = 0.0

### The Loss Curve

The train line is jittery because batch size is 1: each step's gradient comes from a single 8-char window. Real training averages over thousands of windows at once (a "batch"): smoother, GPU-friendly, same model.

In [ ]:
import matplotlib.pyplot as plt

steps = []
vals = []
for s, v in val_history:
    steps.append(s)
    vals.append(v)

plt.figure(figsize=(9, 4))
plt.plot(train_losses, alpha=0.3, label="train loss (per chunk)")
plt.plot(steps, vals, "o-", label="val loss")
plt.axhline(math.log(vocab_size), linestyle="--", color="gray", label="random guessing = ln(24)")
plt.xlabel("training step")
plt.ylabel("loss")
plt.legend()
plt.show()

### Experiment: break the initialization

Why `init_std = 0.08`? Change it in the settings cell, then `Runtime > Run all`:

| init | result (measured, 400 steps) |
|---|---|
| all zeros | loss frozen at exactly ln(24) = 3.178 forever: zero vectors make zero gradients. dead. |
| all ones | numbers compound until a probability hits exactly 0.0 → `log(0)` **crashes** |
| std 1.0 | starts at loss **175** (confidently wrong), still 8.9 after 400 steps |
| std 0.3 | works: 5.2 → 0.69 |
| std 0.08 | works: 3.2 → 0.55 |
| std 0.005 | works: 3.2 → 0.53 |

There is no magic number, only a safe band: small and random. (At 96 layers the band narrows brutally; real models derive the scale from the architecture, e.g. ~1/√d, instead of vibes.)

## 10 · Peek Inside: what attention learned

Each row = a token predicting; bright cells = the earlier tokens it looks at. The empty upper triangle is causality: no peeking at the future.

In [ ]:
prompt = "Peter Pi"
ids = []
for c in prompt:
    ids.append(characters.index(c))
_, attention_maps = gpt_forward(ids, return_attention=True)

grid = []
for row in attention_maps:
    full_row = []
    for j in range(len(ids)):
        if j < len(row):
            full_row.append(row[j])
        else:
            full_row.append(0.0)
    grid.append(full_row)

plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap="Blues")
plt.xticks(range(len(ids)), list(prompt))
plt.yticks(range(len(ids)), list(prompt))
plt.xlabel("attending to (earlier tokens)")
plt.ylabel("current token")
plt.colorbar()
plt.show()

## 11 · Play

In [ ]:
generate_text("A peck of ")
generate_text("If Peter ")
generate_text("pickled ")

In [ ]:
# temperature: <1 plays it safe, >1 gets adventurous
generate_text("Peter Piper ", temperature=0.5)
generate_text("Peter Piper ", temperature=1.0)
generate_text("Peter Piper ", temperature=1.5)

## 12 · Train It on Your Own Text

Swap `txt` in the first code cell (short and repetitive works best, 200-400 characters), then `Runtime > Run all`. Prompts can only use characters that appear in your text.

## Victory Lap

Everything you just built, compressed to 98 lines: [`v1.py`](https://github.com/Infatoshi/yoctogpt/blob/master/v1.py). Read it top to bottom: you now understand every line.

| This notebook | ChatGPT |
|---|---|
| 3,968 parameters | trillions of parameters |
| 8-character context | hundreds of thousands of tokens |
| character tokens | subword tokens (~50k vocabulary) |
| 1 attention head, 1 layer | ~100 heads × ~100 layers |
| 1 window per step | thousands of windows per step (batching) |
| tongue twister | a large chunk of the internet |
| predict the next token | predict the next token |

**Go deeper:** [LLMs from scratch, PyTorch, 6 hrs](https://www.youtube.com/watch?v=UU1WVnMk4E8) · [nanogpt lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY) · [CUDA course, 12 hrs](https://www.youtube.com/watch?v=86FAWCzIe_4) · [backprop on a whiteboard](https://youtu.be/0dbihoMRuyg)

---
## Appendix: Terminology Dictionary

Every scary ML term in this notebook, with the mechanics behind the word:

| Term | What it actually is |
|------|---------------------|
| **Token** | A piece of text swapped for a number, like an arcade token standing in for a coin (for us: one character) |
| **Embedding** | A token's point in 16-dimensional space; "embedded" from separate symbols into continuous numbers so *closeness* exists |
| **Position embedding** | A learned vector meaning "I'm in slot 3"; added because attention has no built-in sense of order |
| **Parameter / Weight** | One adjustable number in the brain (we have 3,968 of them) |
| **Projection / Linear layer** | Matrix × vector: one dot product per row, mapping a vector into a new space (16 numbers in, 24 out) |
| **Forward pass** | Running the model to make a prediction |
| **Loss** | The score of the model's bet: -log(probability it gave the right answer); lower = better |
| **Gradient** | For each parameter: which direction (and how strongly) to nudge it to reduce loss |
| **Backward pass** | Computing all 3,968 gradients via the chain rule |
| **Learning rate** | How big a nudge each step takes |
| **Attention** | A weighted average over earlier tokens, where the model *learns* the weights from content (Q·K scores) |
| **Normalization (RMS)** | Dividing a vector by √(mean of squares) so its typical size is ~1; stops values exploding or vanishing across layers |
| **Nonlinearity / ReLU** | max(0, x): the bend that stops stacked linear layers collapsing into one |
| **Softmax** | exp every score, divide by the sum: scores become probabilities that sum to 1 |
| **Logits** | The raw scores before softmax (one per character) |
| **KV cache** | Stored Key/Value vectors of earlier tokens so generation doesn't recompute them |

Don't memorize these; refer back as needed.